In [1]:
!pwd

/content


In [2]:
# Explicitly installing the missing unsloth_zoo dependency and refreshing core packages
!pip install unsloth_zoo
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.30" "trl<0.16.0" peft accelerate bitsandbytes

  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.15.2
    Uninstalling trl-0.15.2:
      Successfully uninstalled trl-0.15.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-yr08gr6m/unsloth_bf353a7062d949068f8494f0ebc063b1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-yr08gr6m/unsloth_bf353a7062d949068f8494f0ebc063b1
  Resolved https://github.com/unslothai/unsloth.git to commit a2f379314564710a731d4a119a5502c4cbc8e314
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.15.2-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.15.2-py3-none-any.whl (318 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successf

In [3]:
import torch
import gc

# 1. T4 Stability Cleanup
gc.collect()
torch.cuda.empty_cache()

try:
    from unsloth import FastLanguageModel
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainingArguments
    print("Environment Ready: Unsloth and dependencies loaded successfully.")
except ImportError:
    print("Unsloth still not found. Please ensure you have restarted the session (Runtime -> Restart session) and try again.")
    raise

# 2. Optimized Model Loading (Tesla T4 Specific)
max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    gpu_memory_utilization = 0.3, # Large safety margin for T4 VRAM
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Data Preparation
dataset = load_dataset("json", data_files="vitalis_brand_alignment_1k.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)

# 4. SFT Configuration (Fixes for Transformers 5.x)
print("\nCommencing Supervised Fine-Tuning for Vitalis AI...")

training_args = SFTConfig(
    output_dir = "outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 1,
    optim = "paged_adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    report_to = "none",
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
)

# Force-removal of internal keys that cause TypeError in certain Transformers versions
for key in ["push_to_hub_token", "hub_token", "push_to_hub_model_id"]:
    if hasattr(training_args, key): delattr(training_args, key)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

trainer.train()
print("\nSuccess! Vitalis AI behavioral alignment is complete.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
Environment Ready: Unsloth and dependencies loaded successfully.
==((====))==  Unsloth 2026.5.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.



Commencing Supervised Fine-Tuning for Vitalis AI...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.181189
2,4.137531
3,3.976542
4,3.619822
5,3.316576
6,3.026343
7,2.743691
8,2.429351
9,2.150499
10,1.779366


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.



Success! Vitalis AI behavioral alignment is complete.


In [4]:
# 1. Place the optimized Unsloth model into inference mode
FastLanguageModel.for_inference(model)

# 2. Frame a tough, emotionally stressed customer query
test_messages = [
    {
        "role": "system",
        "content": (
            "You are Vitalis AI, a warm, plain-spoken health insurance guide. "
            "Your mission is to explain insurance simply, with radical transparency and empathy. "
            "Never use bureaucratic jargon or escape clauses like 'pursuant to section 4B'. "
            "Break down costs in plain english and actively reassure the customer."
        )
    },
    {
        "role": "user",
        "content": "Why didn't you pay for my doctor visit? I got a confusing bill for $300 and I am super stressed."
    }
]

# 3. Apply the Llama 3 format structure
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 4. Generate the brand-aligned response
outputs = model.generate(input_ids=inputs, max_new_tokens=250, use_cache=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 5. Clean print the response
print("\n--- Vitalis AI Output Verification ---")
print(response.split("assistant")[-1].strip())

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn


--- Vitalis AI Output Verification ---
Insurance statements look like a puzzle, so let's break down that $300 in plain English. Your copay is what you pay at the door, but your plan has a standard deductible for laboratory or imaging tests. That $300 went directly toward your annual deductible balance. The good news? You are now incredibly close to hitting that limit, meaning Vitalis picks up almost the entire bill for the rest of the year.


In [7]:
import torch
import gc

# Clear previous model and free up GPU memory
try:
    if 'model' in locals():
        model = None
    if 'tokenizer' in locals():
        tokenizer = None
    torch.cuda.synchronize() # Ensure all CUDA ops are done
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize() # Ensure cache clearing is done
except NameError:
    pass # If model/tokenizer were never defined, this won't be an issue

from unsloth import FastLanguageModel
print("GPU detected and Unsloth loaded correctly.")

max_seq_length = 512
dtype = None
load_in_4bit = True

# Load the original base model
base_model_name = "unsloth/meta-Llama-3.1-8B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # Explicitly map all layers to GPU 0 to prevent CPU offloading
    device_map = {"": 0},
)

# Re-initialize the PEFT model structure with the same parameters as training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)
# Load the LoRA adapter weights from the 'outputs/checkpoint-60' directory
model.load_adapter("outputs/checkpoint-60", "outputs")

print("Model re-loaded from checkpoint. Starting GGUF export...")

# Merge and export to GGUF format
model.save_pretrained_gguf(
    "vitalis_brand_model",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("\nSuccess! Your GGUF model is ready in the 'vitalis_brand_model' folder.")

GPU detected and Unsloth loaded correctly.
==((====))==  Unsloth 2026.5.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Model re-loaded from checkpoint. Starting GGUF export...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in vitalis_brand_model/tokenizer_config.json.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [13:15<00:00, 198.91s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:19<00:00, 124.96s/it]


Unsloth: Merge process complete. Saved to `/content/vitalis_brand_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['vitalis_brand_model_gguf/Meta-Llama-3.1-8B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['vitalis_brand_model_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model vitalis_brand_model_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to vitalis_brand_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f vitalis_brand_model_gguf/Modelfile

Success! Your GGUF model is ready in the 'vitalis_brand_model' folder.
